# ETL Spotify — Extracción de Métricas Empresariales y B2C

Este notebook implementa un pipeline ETL que consume la **API de Spotify** a través de `spotipy`
y genera dos datasets para alimentar dashboards de **Business Intelligence (BI)** y **recomendaciones B2C**.

## Flujo del pipeline

1. **Autenticación** via `Client Credentials` de Spotify (credenciales desde `.env`).
2. **Extracción** — Por cada género musical:
   - Top 10 artistas (nombre, id, popularidad, seguidores, imagen, géneros secundarios).
   - Top 3 pistas por artista (nombre, id, popularidad, preview_url, duración, explicititud, imagen de álbum).
3. **Demografía sintética** — Columnas de edad y sexo simuladas con valores realistas por género.
4. **Métricas empresariales** — KPIs compuestos para decisiones de inversión y seguimiento de metas.
5. **Exportación** — Dos CSVs listos para consumir desde Tableau / Streamlit / Power BI.

## Salidas generadas

| Archivo | Contenido |
|---|---|
| `data/spotify/spotify_business_metrics.csv` | KPIs de mercado, inversión y metas por género (7 filas, 22 columnas) |
| `data/spotify/spotify_b2c_recommendations.csv` | Top artistas y canciones por género (archivo plano) |

## KPIs de negocio incluidos

| Grupo | KPIs |
|---|---|
| **Catálogo** | `artistas_unicos`, `total_canciones_encontradas`, `duracion_promedio_ms`, `porcentaje_explicitos`, `porcentaje_con_preview` |
| **Alcance** | `popularidad_max`, `popularidad_min`, `diversidad_generos_secundarios` |
| **Comerciabilidad** | `indice_comerciabilidad` (0–100), `nivel_madurez_mercado` |
| **Inversión** | `score_inversion` (0–100), `riesgo_inversion`, `inversion_recomendada_usd`, `roi_estimado_pct` |
| **Demografía** | `edad_promedio_oyente`, `porcentaje_hombres`, `porcentaje_mujeres` |
| **Metas** | `potencial_ganancia_usd`, `meta_ingresos_usd`, `cumplimiento_meta_pct`, `tendencia_crecimiento_pct` |


In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import spotipy
from dotenv import load_dotenv
from spotipy.exceptions import SpotifyException
from spotipy.oauth2 import SpotifyClientCredentials

warnings.filterwarnings("ignore")
print("Librerias cargadas OK")


In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

# Cargar credenciales desde archivo .env (en la raiz del proyecto)
load_dotenv()
CLIENT_ID = os.environ.get("SPOTIPY_CLIENT_ID", "")
CLIENT_SECRET = os.environ.get("SPOTIPY_CLIENT_SECRET", "")

if not CLIENT_ID or not CLIENT_SECRET:
    print("ADVERTENCIA: Credenciales no encontradas en .env")
    print("Crear archivo .env con: SPOTIPY_CLIENT_ID=xxx y SPOTIPY_CLIENT_SECRET=yyy")

# Lista de generos musicales a consultar
GENRES = ["vallenato", "jazz", "clasica", "pop", "rock", "electronica", "hiphop"]

# Limites de extraccion
TOP_N_ARTISTS = 10
TOP_N_TRACKS = 3

# Directorio de salida
DATA_DIR = Path("../data/spotify")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Directorio de salida: {DATA_DIR.resolve()}")

# Base de datos demograficos sinteticos por genero
DEMOGRAPHIC_BASE = {
    "vallenato":   {"edad": 30, "hombres": 55, "mujeres": 42, "ganancia": 450_000},
    "jazz":        {"edad": 47, "hombres": 52, "mujeres": 43, "ganancia": 310_000},
    "clasica":     {"edad": 52, "hombres": 44, "mujeres": 51, "ganancia": 280_000},
    "pop":         {"edad": 25, "hombres": 38, "mujeres": 57, "ganancia": 950_000},
    "rock":        {"edad": 34, "hombres": 62, "mujeres": 33, "ganancia": 720_000},
    "electronica": {"edad": 28, "hombres": 56, "mujeres": 39, "ganancia": 540_000},
    "hiphop":      {"edad": 23, "hombres": 68, "mujeres": 28, "ganancia": 880_000},
}
print(f"{len(GENRES)} generos configurados: {GENRES}")


In [ ]:
def setup_spotify() -> spotipy.Spotify:
    """Inicializa y devuelve un cliente autenticado de Spotify."""
    try:
        auth_manager = SpotifyClientCredentials(
            client_id=CLIENT_ID,
            client_secret=CLIENT_SECRET,
        )
        sp = spotipy.Spotify(auth_manager=auth_manager)
        # Prueba rapida de conectividad
        sp.search(q="test", type="artist", limit=1)
        print("Autenticacion exitosa contra la API de Spotify")
        return sp
    except SpotifyException as e:
        print(f"Error de autenticacion: {e}")
        raise
    except Exception as e:
        print(f"Error inesperado al conectar con Spotify: {e}")
        raise

In [ ]:
def _api_call_with_retry(
    func: callable,
    *args,
    max_retries: int = 5,
    **kwargs,
) -> dict | None:
    """Ejecuta una llamada a la API de Spotify con reintento
    exponencial ante errores 429 (Rate Limit).

    Parametros
    ----------
    func : callable
        Metodo de spotipy.Spotify a invocar.
    max_retries : int
        Numero maximo de reintentos.

    Retorna
    -------
    dict | None
        Respuesta de la API, o None si falla tras todos los reintentos.
    """
    for attempt in range(1, max_retries + 1):
        try:
            result = func(*args, **kwargs)
            time.sleep(1.0)
            return result
        except SpotifyException as e:
            if e.http_status == 429:
                retry_after = int(
                    getattr(e, "headers", {}).get("Retry-After", 2**attempt)
                )
                print(
                    f"  Rate limit (429). Reintentando en {retry_after}s "
                    f"(intento {attempt}/{max_retries})"
                )
                time.sleep(retry_after)
            else:
                print(f"  Error Spotify ({e.http_status}): {e.msg}")
                return None
        except Exception as e:
            print(f"  Error inesperado: {e}")
            return None
    print(f"  Se agotaron los reintentos ({max_retries})")
    return None

In [ ]:
def search_artists_by_genre(
    sp: spotipy.Spotify,
    genre: str,
    limit: int = 10,
) -> list[dict]:
    """Busca los artistas mas relevantes para un genero dado.

    Primero intenta genre:\"genre\", si da 0 resultados hace un
    fallback a busqueda por nombre del genero.

    El search basico no incluye popularity ni followers, por lo que
    se hace una llamada adicional por artista al endpoint de detalle.
    """
    try:
        # Intento 1: busqueda por genero exacto
        results = _api_call_with_retry(
            sp.search, q=f'genre:"{genre}"', type="artist", limit=limit
        )
        artists = []
        if results:
            artists = results.get("artists", {}).get("items", [])

        # Intento 2: fallback por nombre del genero
        if not artists:
            print(f"  Sin resultados con genre:, probando busqueda por nombre...")
            results = _api_call_with_retry(
                sp.search, q=genre, type="artist", limit=limit
            )
            if results:
                artists = results.get("artists", {}).get("items", [])

        parsed = []
        for a in artists:
            full = _api_call_with_retry(sp.artist, a["id"])
            if full is None:
                continue
            parsed.append({
                "id": a["id"],
                "name": a["name"],
                "popularity": full.get("popularity") or 0,
                "followers": (full.get("followers") or {}).get("total", 0),
                "image_url": a["images"][0]["url"] if a["images"] else "",
                "genres": full.get("genres", []),
            })
        print(f"  {genre}: {len(parsed)} artistas encontrados")
        return parsed
    except Exception as e:
        print(f"  Error buscando artistas para {genre}: {e}")
        return []


In [ ]:
def get_top_tracks(
    sp: spotipy.Spotify,
    artist_name: str,
    limit: int = 3,
) -> list[dict]:
    """Obtiene las pistas mas populares de un artista via search.\n    artist_top_tracks requiere auth de usuario (da 403 con Client Credentials),\n    por lo que se usa sp.search con filtro artist:.\n    """
    try:
        results = _api_call_with_retry(
            sp.search, q=f'artist:"{artist_name}"', type="track", limit=limit,
        )
        if results is None:
            return []

        tracks = results.get("tracks", {}).get("items", [])
        # Filtrar tracks cuyo artista principal coincida exactamente
        top = [
            t for t in tracks
            if any(a["name"].lower() == artist_name.lower()
                   for a in t.get("artists", []))
        ][:limit]

        parsed = []
        for t in top:
            parsed.append({
                "id": t["id"],
                "name": t["name"],
                "popularity": t.get("popularity") or 0,
                "duration_ms": t.get("duration_ms") or 0,
                "explicit": t.get("explicit", False),
                "preview_url": t.get("preview_url") or "",
                "album_image": (
                    t["album"]["images"][0]["url"]
                    if t["album"].get("images")
                    else ""
                ),
            })
        return parsed
    except Exception as e:
        print(f"    Error obteniendo tracks de {artist_name}: {e}")
        return []


In [ ]:
def extract_b2c_data(
    sp: spotipy.Spotify,
    genres: list[str],
) -> pd.DataFrame:
    """Construye el dataset plano de recomendaciones B2C.\n    Para cada genero extrae top N artistas y top M pistas por artista,\n    consolidando todo en un DataFrame fila por cancion.\n    """
    rows = []

    for genre in genres:
        print(f"\nProcesando genero: {genre}")
        artists = search_artists_by_genre(sp, genre, limit=TOP_N_ARTISTS)

        for artist in artists:
            tracks = get_top_tracks(sp, artist["name"], limit=TOP_N_TRACKS)
            for track in tracks:
                rows.append({
                    "genero": genre,
                    "artista_nombre": artist["name"],
                    "artista_id": artist["id"],
                    "popularidad_artista": artist["popularity"],
                    "seguidores_artista": artist["followers"],
                    "imagen_artista": artist["image_url"],
                    "generos_artista": ";".join(artist.get("genres", [])),
                    "cancion_nombre": track["name"],
                    "cancion_id": track["id"],
                    "popularidad_cancion": track["popularity"],
                    "duracion_ms": track["duration_ms"],
                    "es_explicito": track["explicit"],
                    "url_preview": track["preview_url"],
                    "imagen_album": track["album_image"],
                })
            print(f"  {artist['name']}: {len(tracks)} pistas")

    df = pd.DataFrame(rows)
    print(f"\nB2C: {len(df)} registros generados")
    return df


In [ ]:
def generate_demographics(genre: str, seed: int | None = None) -> dict:
    """Genera datos demograficos sinteticos realistas para un genero.

    Aplica ruido gaussiano sobre los valores base para simular
    variabilidad natural en cada extraccion.

    Parametros
    ----------
    genre : str
        Nombre del genero musical.
    seed : int | None
        Semilla opcional para reproducibilidad.

    Retorna
    -------
    dict
        Edad promedio, porcentajes de genero y ganancia potencial.
    """
    rng = np.random.default_rng(seed)
    base = DEMOGRAPHIC_BASE[genre]

    edad = max(14, rng.normal(loc=base["edad"], scale=2.0))
    hombres = rng.normal(loc=base["hombres"], scale=3.0)
    mujeres = rng.normal(loc=base["mujeres"], scale=3.0)

    # Asegurar que los porcentajes sumen ~100
    total = hombres + mujeres
    if total > 0:
        hombres = max(0, hombres / total * 100)
        mujeres = max(0, mujeres / total * 100)

    ganancia = max(100_000, rng.normal(loc=base["ganancia"], scale=50_000))

    return {
        "edad_promedio_oyente": round(edad, 1),
        "porcentaje_hombres": round(hombres, 1),
        "porcentaje_mujeres": round(mujeres, 1),
        "potencial_ganancia_usd": int(ganancia),
    }

In [ ]:
def extract_business_metrics(
    genres: list[str],
    df_b2c: pd.DataFrame,
) -> pd.DataFrame:
    """Construye el dataset de metricas de negocio por genero para BI empresarial.

    Combina datos agregados reales (del df_b2c) con demograficos sinteticos y KPIs
    diseñados para toma de decisiones de inversion y medicion de metas.

    KPIs de catalogo:
      - artistas_unicos, duracion_promedio_ms, porcentaje_explicitos
      - popularidad_max, popularidad_min, diversidad_generos_secundarios
      - total_canciones_encontradas, porcentaje_con_preview

    KPIs de mercado:
      - indice_comerciabilidad (0-100): ponderacion de popularidad, preview y contenido
      - nivel_madurez_mercado: Nicho / Emergente / Crecimiento / Maduro

    KPIs de inversion:
      - score_inversion (0-100): indice compuesto de atractivo para invertir
      - riesgo_inversion: Bajo / Medio / Alto
      - inversion_recomendada_usd: capital sugerido segun riesgo y potencial
      - roi_estimado_pct: retorno estimado sobre la inversion recomendada

    KPIs de metas:
      - meta_ingresos_usd: objetivo de ingresos (120% del potencial base)
      - cumplimiento_meta_pct: avance hacia la meta (potencial / meta * 100)
      - tendencia_crecimiento_pct: proyeccion de crecimiento anclada al score
    """
    rows = []
    FACTORES_RIESGO = {"Bajo": 0.25, "Medio": 0.15, "Alto": 0.08}

    for genre in genres:
        df_genre = df_b2c[df_b2c["genero"] == genre]

        if df_genre.empty:
            print(f"  {genre}: sin datos, usando defaults")
            rows.append({
                "genero": genre,
                "artistas_unicos": 0,
                "duracion_promedio_ms": 0.0,
                "porcentaje_explicitos": 0.0,
                "popularidad_max": 0,
                "popularidad_min": 0,
                "diversidad_generos_secundarios": 0.0,
                "total_canciones_encontradas": 0,
                "porcentaje_con_preview": 0.0,
                "indice_comerciabilidad": 0.0,
                "score_inversion": 0.0,
                "riesgo_inversion": "Alto",
                "nivel_madurez_mercado": "Nicho",
                "inversion_recomendada_usd": 0,
                "roi_estimado_pct": 0.0,
                **generate_demographics(genre),
                "meta_ingresos_usd": 0,
                "cumplimiento_meta_pct": 0.0,
                "tendencia_crecimiento_pct": 0.0,
            })
            continue

        # — Metricas internas (no se exportan, usadas para calculo de KPIs) —
        pop_prom = float(df_genre["popularidad_artista"].mean())

        # — Metricas de catalogo —
        artistas_unicos = int(df_genre["artista_nombre"].nunique())
        dur_prom = float(df_genre["duracion_ms"].mean())
        pct_explicit = float(df_genre["es_explicito"].mean() * 100)
        pop_max = int(df_genre["popularidad_artista"].max())
        pop_min = int(df_genre["popularidad_artista"].min())

        gen_counts = df_genre["generos_artista"].dropna().apply(
            lambda x: len(x.split(";")) if x else 0
        )
        div_generos = float(gen_counts.mean()) if len(gen_counts) > 0 else 0.0

        total_canciones = len(df_genre)
        pct_preview = float(
            df_genre["url_preview"].apply(lambda x: 1 if x and x.strip() else 0).mean() * 100
        ) if total_canciones > 0 else 0.0

        # — Indices compuestos de mercado —
        # Comerciabilidad: popularidad de techo + disponibilidad de preview + bajo contenido explicito
        indice_comerciabilidad = round(
            (pop_max * 0.40) + (pct_preview * 0.40) + ((100 - pct_explicit) * 0.20),
            1,
        )

        # Score de inversion (0-100): comerciabilidad + popularidad promedio + diversidad + cobertura de artistas
        score_inversion = round(min(100.0,
            indice_comerciabilidad * 0.40 +
            pop_prom * 0.30 +
            min(100.0, div_generos / 10.0 * 100) * 0.15 +
            min(100.0, artistas_unicos / 10.0 * 100) * 0.15
        ), 1)

        # Nivel de riesgo segun score
        if score_inversion >= 65:
            riesgo = "Bajo"
        elif score_inversion >= 40:
            riesgo = "Medio"
        else:
            riesgo = "Alto"

        # Madurez del mercado segun popularidad maxima alcanzada
        if pop_max >= 80:
            madurez = "Maduro"
        elif pop_max >= 60:
            madurez = "Crecimiento"
        elif pop_max >= 40:
            madurez = "Emergente"
        else:
            madurez = "Nicho"

        demo = generate_demographics(genre)

        # Inversion recomendada: fraccion del potencial segun riesgo
        inversion_rec = int(demo["potencial_ganancia_usd"] * FACTORES_RIESGO[riesgo])
        roi_estimado = round(
            (demo["potencial_ganancia_usd"] - inversion_rec) / max(1, inversion_rec) * 100, 1
        )

        # Metas: objetivo al 120% del potencial base
        meta_ingresos = int(demo["potencial_ganancia_usd"] * 1.20)
        cumplimiento_meta = round(
            demo["potencial_ganancia_usd"] / meta_ingresos * 100, 1
        )

        # Tendencia de crecimiento sintetica: anclada al score, con variabilidad por genero
        rng_g = np.random.default_rng(hash(genre) % 2**32)
        base_trend = (score_inversion - 50) * 0.5
        tendencia = round(float(max(-30.0, min(50.0, rng_g.normal(loc=base_trend, scale=5.0)))), 1)

        rows.append({
            "genero": genre,
            "artistas_unicos": artistas_unicos,
            "duracion_promedio_ms": round(dur_prom, 0),
            "porcentaje_explicitos": round(pct_explicit, 1),
            "popularidad_max": pop_max,
            "popularidad_min": pop_min,
            "diversidad_generos_secundarios": round(div_generos, 2),
            "total_canciones_encontradas": total_canciones,
            "porcentaje_con_preview": round(pct_preview, 1),
            "indice_comerciabilidad": indice_comerciabilidad,
            "score_inversion": score_inversion,
            "riesgo_inversion": riesgo,
            "nivel_madurez_mercado": madurez,
            "inversion_recomendada_usd": inversion_rec,
            "roi_estimado_pct": roi_estimado,
            **demo,
            "meta_ingresos_usd": meta_ingresos,
            "cumplimiento_meta_pct": cumplimiento_meta,
            "tendencia_crecimiento_pct": tendencia,
        })

    df = pd.DataFrame(rows)
    print(f"\nBusiness Metrics: {len(df)} generos procesados")
    return df


In [ ]:
def export_datasets(
    df_business: pd.DataFrame,
    df_b2c: pd.DataFrame,
    data_dir: Path = DATA_DIR,
) -> None:
    """Exporta ambos DataFrames a archivos CSV.

    Parametros
    ----------
    df_business : pd.DataFrame
        Dataset de metricas de negocio.
    df_b2c : pd.DataFrame
        Dataset de recomendaciones B2C.
    data_dir : Path
        Directorio de salida.
    """
    business_path = data_dir / "spotify_business_metrics.csv"
    b2c_path = data_dir / "spotify_b2c_recommendations.csv"

    df_business.to_csv(business_path, index=False)
    print(f"Business Metrics exportado: {business_path.resolve()}")

    df_b2c.to_csv(b2c_path, index=False)
    print(f"B2C Recommendations exportado: {b2c_path.resolve()}")

    print(f"\nResumen:")
    print(f"  Business Metrics: {len(df_business)} filas, "
          f"{list(df_business.columns)}")
    print(f"  B2C Recommendations: {len(df_b2c)} filas, "
          f"{list(df_b2c.columns)}")

In [ ]:
def main() -> None:
    """Orquestador principal del pipeline ETL."""
    print("=" * 55)
    print("  ETL SPOTIFY - INICIANDO PIPELINE")
    print("=" * 55)

    # 1. Autenticacion
    print("\n[1/4] Autenticando...")
    try:
        sp = setup_spotify()
    except Exception:
        print("No se pudo autenticar. Revisa tu .env con SPOTIPY_CLIENT_ID y SPOTIPY_CLIENT_SECRET.")
        return

    # 2. Extraccion B2C
    print("\n[2/4] Extrayendo datos B2C...")
    try:
        df_b2c = extract_b2c_data(sp, GENRES)
    except Exception as e:
        print(f"Error en extraccion B2C: {e}")
        df_b2c = pd.DataFrame()

    # 3. Extraccion Business Metrics (usa df_b2c, no duplica llamadas API)
    print("\n[3/4] Calculando metricas de negocio...")
    try:
        df_business = extract_business_metrics(GENRES, df_b2c)
    except Exception as e:
        print(f"Error en extraccion Business: {e}")
        df_business = pd.DataFrame()

    # 4. Exportacion
    print("\n[4/4] Exportando datasets...")
    try:
        export_datasets(df_business, df_b2c)
    except Exception as e:
        print(f"Error en exportacion: {e}")

    print("\n" + "=" * 55)
    print("  PIPELINE ETL COMPLETADO")
    print("=" * 55)


In [ ]:
if __name__ == "__main__":
    main()